In [10]:
import json
import random

# Bind the specific tasks_id directly to the actual defect and department
TASK_CATALOG = {
    "TMS-1": {"dept": "TMS", "defect": "Rail Fracture"},
    "TMS-2": {"dept": "TMS", "defect": "Fresh Track Relaying"},
    "TMS-3": {"dept": "TMS", "defect": "TQI Exceedence"},
    "TMS-4": {"dept": "TMS", "defect": "Ballast Renewal"},
    "TMS-5": {"dept": "TMS", "defect": "Sleeper Replacement"},
    "TDMS-1": {"dept": "TDMS", "defect": "OHE Wire Tensioning"},
    "TDMS-2": {"dept": "TDMS", "defect": "Tower Wagon Foot Patrol"},
    "TDMS-3": {"dept": "TDMS", "defect": "Contact Wire Wear"},
    "TDMS-4": {"dept": "TDMS", "defect": "Power Block"},
    "TDMS-5": {"dept": "TDMS", "defect": "Insulator Cleaning"},
    "SMMS-1": {"dept": "SMMS", "defect": "Point Motor Failure"},
    "SMMS-2": {"dept": "SMMS", "defect": "Axle Counter Error"},
    "SMMS-3": {"dept": "SMMS", "defect": "Interlocking Downgrade"}
}

def create_mock_json(filename="mock_simulation_data.json", num_tasks=30):
    sections = ["HWH_to_BWN", "BWN_to_BHP", "BHP_to_MLDT", "MLDT_to_KNE", "KNE_to_NJP"]
    durations = [45, 60, 90, 120, 180, 240]
    frequencies = ["Weekly", "Monthly", "Special"]

    maintenance_queue = []
    task_keys = list(TASK_CATALOG.keys())

    for i in range(1, num_tasks + 1):
        # Pick a valid task ID (e.g., "TMS-4")
        chosen_task_id = random.choice(task_keys)
        task_info = TASK_CATALOG[chosen_task_id]

        task = {
            "serial_number": i,  # Purely sequential identifier
            "tasks_id": chosen_task_id,
            "department": task_info["dept"],
            "defect_type": task_info["defect"], # Injects the defect name for the priority math
            "section_id": random.choice(sections),
            "work_duration_mins": random.choice(durations),
            "maintenance_frequency": random.choice(frequencies) # Injects the routine frequency
        }
        maintenance_queue.append(task)

    payload = {
        "route": "HWH_to_NJP",
        "maintenance_queue": maintenance_queue
    }

    with open(filename, "w") as f:
        json.dump(payload, f, indent=2)

if __name__ == "__main__":
    create_mock_json()
    print("mock_simulation_data.json has been generated successfully!")

mock_simulation_data.json has been generated successfully!


In [11]:
import json
import sys

# --- MAPPINGS & CONFIGURATION ---

DEPT_MAP = {
    "Engineering": "TMS", "TMS": "TMS",
    "Traction": "TDMS", "TDMS": "TDMS",
    "Signalling": "SMMS", "SMMS": "SMMS"
}

DEFECT_TIER_MAP = {
    # Tier 1 (Emergencies)
    "Rail Fracture": "Tier 1",
    "Power Block": "Tier 1",
    "Point Motor Failure": "Tier 1",
    "Interlocking Downgrade": "Tier 1",
    "Axle Counter Error": "Tier 1",

    # Tier 2 (Major Bottlenecks)
    "Fresh Track Relaying": "Tier 2",
    "Ballast Renewal": "Tier 2",
    "TQI Exceedence": "Tier 2",

    # Tier 3 (Routine/Preventative)
    "Sleeper Replacement": "Tier 3",
    "OHE Wire Tensioning": "Tier 3",
    "Tower Wagon Foot Patrol": "Tier 3",
    "Contact Wire Wear": "Tier 3",
    "Insulator Cleaning": "Tier 3"
}

BASE_SCORES = {
    "TMS":  {"Tier 1": 10000, "Tier 2": 5000, "Tier 3": 1000},
    "SMMS": {"Tier 1": 8000,  "Tier 2": 4000, "Tier 3": 800},
    "TDMS": {"Tier 1": 7000,  "Tier 2": 3500, "Tier 3": 700}
}

# (Restricted Speed in km/h, Worksite Length in km)
DEFECT_PHYSICS = {
    "Rail Fracture": (0, 1.5),
    "Power Block": (0, 2.0),
    "Point Motor Failure": (15, 1.5),
    "Interlocking Downgrade": (15, 1.5),
    "Axle Counter Error": (15, 1.5),
    "Fresh Track Relaying": (30, 4.0),
    "Ballast Renewal": (30, 4.0),
    "TQI Exceedence": (50, 7.5),
    "Sleeper Replacement": (130, 2.0),
    "OHE Wire Tensioning": (130, 2.0),
    "Contact Wire Wear": (130, 2.0),
    "Insulator Cleaning": (130, 2.0),
    "Tower Wagon Foot Patrol": (130, 2.0)
}

# --- PRIORITY CALCULATION ENGINE ---

def calculate_priority(task, current_month=9):
    """
    Calculates priority based on available keys, defaulting missing data to routine values.
    """
    # 1. Map Department
    raw_dept = task.get("department", "TMS")
    dept = DEPT_MAP.get(raw_dept, raw_dept)

    # 2. Extract defect type and map to tier dynamically
    defect = task.get("defect_type", "Sleeper Replacement")
    tier = DEFECT_TIER_MAP.get(defect, "Tier 3") # Defaults to Tier 3 if defect is unknown

    # HARD OVERRIDE: TMS Tier 1
    if dept == "TMS" and tier == "Tier 1":
        return 100000

    # 3. Base Score & Monsoon Multiplier based on dynamically assigned tier
    base_score = BASE_SCORES.get(dept, BASE_SCORES["TMS"]).get(tier, 1000)

    if current_month in [6, 7, 8, 9] and dept == "TMS":
        base_score = int(base_score * 1.2)

    # 4. Aging Bonus (Fallback to 0)
    days_ignored = task.get("days_ignored", 0)
    aging_bonus = days_ignored * 100

    # 6. Kinetic Physics / Speed Impact
    restricted_speed, worksite_length_km = DEFECT_PHYSICS.get(defect, (130, 2.0))
    normal_speed = 130

    if restricted_speed == 0:
        return 100000
    elif restricted_speed < normal_speed:
        normal_time_hrs = worksite_length_km / normal_speed
        restricted_time_hrs = worksite_length_km / restricted_speed
        time_lost_mins = ((restricted_time_hrs - normal_time_hrs) * 60) + 2
        time_impact = int(time_lost_mins)
    else:
        time_impact = 0

    # 7. Frequency Bonus (Weekly / Monthly Priority)
    frequency = str(task.get("maintenance_frequency", "Ad-hoc")).strip().capitalize()
    frequency_bonus = 0
    if frequency == "Weekly":
        frequency_bonus = 500  # Pushes it to the top of the routine Tier 3 queue
    elif frequency == "Monthly":
        frequency_bonus = 300  # Elevated, but yields to weekly tasks

    # 8. Final Tabulation
    final_score = base_score + aging_bonus + time_impact + frequency_bonus

    if final_score == sys.maxsize:
      return 100000
    else:
      return final_score

# --- EXECUTION ---

if __name__ == "__main__":
    input_file = "mock_simulation_data.json"
    output_file = "prioritized_tasks.json"

    try:
        with open(input_file, "r") as f:
            data = json.load(f)

        queue = data.get("maintenance_queue", [])
        print(f"Loaded {len(queue)} tasks. Applying priority scoring...\n")

        for task in queue:
            score = calculate_priority(task)
            task["priority_score"] = score

            # Formatting for display
            display_score = "MAX (Absolute Priority)" if score == sys.maxsize else score
            dept = task.get("department", "Unknown")
            sec = task.get("section_id", "Unknown")
            tid = task.get("tasks_id", task.get("task_id", "N/A"))
            defect = task.get("defect_type", "Sleeper Replacement")
            tier = DEFECT_TIER_MAP.get(defect, "Tier 3")
            freq = str(task.get("maintenance_frequency", "Ad-hoc")).strip().capitalize()

            print(f"Task: {tid:<10} | Tier: {tier:<6} | Freq: {freq:<8} | Section: {sec:<12} | Score: {display_score}")

        # --- NEW CODE: Save the updated payload to a new file ---
        with open(output_file, "w") as out_file:
            json.dump(data, out_file, indent=2)

        print(f"\nSuccess! Updated queue saved to '{output_file}'.")

    except FileNotFoundError:
        print(f"Error: '{input_file}' not found. Please generate it first.")

Loaded 30 tasks. Applying priority scoring...

Task: TDMS-2     | Tier: Tier 3 | Freq: Weekly   | Section: HWH_to_BWN   | Score: 1200
Task: TMS-1      | Tier: Tier 1 | Freq: Monthly  | Section: HWH_to_BWN   | Score: 100000
Task: SMMS-1     | Tier: Tier 1 | Freq: Special  | Section: BWN_to_BHP   | Score: 8007
Task: TMS-1      | Tier: Tier 1 | Freq: Special  | Section: BHP_to_MLDT  | Score: 100000
Task: SMMS-3     | Tier: Tier 1 | Freq: Monthly  | Section: KNE_to_NJP   | Score: 8307
Task: TDMS-2     | Tier: Tier 3 | Freq: Monthly  | Section: BWN_to_BHP   | Score: 1000
Task: SMMS-3     | Tier: Tier 1 | Freq: Special  | Section: KNE_to_NJP   | Score: 8007
Task: TDMS-4     | Tier: Tier 1 | Freq: Monthly  | Section: KNE_to_NJP   | Score: 100000
Task: TMS-4      | Tier: Tier 2 | Freq: Monthly  | Section: BWN_to_BHP   | Score: 6308
Task: TMS-3      | Tier: Tier 2 | Freq: Monthly  | Section: HWH_to_BWN   | Score: 6307
Task: TDMS-5     | Tier: Tier 3 | Freq: Monthly  | Section: HWH_to_BWN   | Sc

In [12]:
import json
import sys
from datetime import datetime, timezone

# --- MAPPINGS & CONFIGURATION ---

DEPT_MAP = {
    "Engineering": "TMS", "TMS": "TMS",
    "Traction": "TDMS", "TDMS": "TDMS",
    "Signalling": "SMMS", "SMMS": "SMMS"
}

DEFECT_TIER_MAP = {
    "Rail Fracture": "Tier 1", "Power Block": "Tier 1", "Point Motor Failure": "Tier 1",
    "Interlocking Downgrade": "Tier 1", "Axle Counter Error": "Tier 1",
    "Fresh Track Relaying": "Tier 2", "Ballast Renewal": "Tier 2", "TQI Exceedence": "Tier 2",
    "Sleeper Replacement": "Tier 3", "OHE Wire Tensioning": "Tier 3",
    "Tower Wagon Foot Patrol": "Tier 3", "Contact Wire Wear": "Tier 3", "Insulator Cleaning": "Tier 3"
}

BASE_SCORES = {
    "TMS":  {"Tier 1": 10000, "Tier 2": 5000, "Tier 3": 1000},
    "SMMS": {"Tier 1": 8000,  "Tier 2": 4000, "Tier 3": 800},
    "TDMS": {"Tier 1": 7000,  "Tier 2": 3500, "Tier 3": 700}
}

DEFECT_PHYSICS = {
    "Rail Fracture": (0, 1.5), "Power Block": (0, 2.0), "Point Motor Failure": (15, 1.5),
    "Interlocking Downgrade": (15, 1.5), "Axle Counter Error": (15, 1.5),
    "Fresh Track Relaying": (30, 4.0), "Ballast Renewal": (30, 4.0),
    "TQI Exceedence": (50, 7.5), "Sleeper Replacement": (130, 2.0),
    "OHE Wire Tensioning": (130, 2.0), "Contact Wire Wear": (130, 2.0),
    "Insulator Cleaning": (130, 2.0), "Tower Wagon Foot Patrol": (130, 2.0)
}

# Add an empty density dict to prevent crashes if not loaded
SECTION_DENSITY = {}

# --- PRIORITY CALCULATION ENGINE ---

def calculate_priority(task, current_month=9):
    raw_dept = task.get("department", "TMS")
    dept = DEPT_MAP.get(raw_dept, raw_dept)
    current_time = datetime.now(timezone.utc)

    defect = task.get("defect_type", "Sleeper Replacement")
    tier = task.get("tier", DEFECT_TIER_MAP.get(defect, "Tier 3"))

    # HARD OVERRIDE: TMS Tier 1
    if dept == "TMS" and tier == "Tier 1":
        return 100000

    base_score = BASE_SCORES.get(dept, BASE_SCORES["TMS"]).get(tier, 1000)

    if current_month in [6, 7, 8, 9] and dept == "TMS":
        base_score = int(base_score * 1.2)

    # --- 2. DYNAMIC AGING BONUS ---
    current_time = datetime.now(timezone.utc)
    request_date_str = task.get("request_date_iso")
    days_ignored = 0

    if request_date_str:
        try:
            req_str_fixed = request_date_str.replace("Z", "+00:00")
            request_date = datetime.fromisoformat(req_str_fixed)

            # Calculate days passed. max(0, ...) prevents negative scores if dates are weird.
            days_ignored = max(0, (current_time - request_date).total_seconds() / 86400.0)
        except ValueError:
            days_ignored = 0 # Fallback if datetime string is corrupted

    aging_bonus = int(days_ignored) * 100

    restricted_speed, worksite_length_km = DEFECT_PHYSICS.get(defect, (130, 2.0))
    normal_speed = 130

    if restricted_speed == 0:
        return 100000
    elif restricted_speed < normal_speed:
        normal_time_hrs = worksite_length_km / normal_speed
        restricted_time_hrs = worksite_length_km / restricted_speed
        time_lost_mins = ((restricted_time_hrs - normal_time_hrs) * 60) + 2
        time_impact = int(time_lost_mins * 100)
    else:
        time_impact = 0

    # --- NEW DYNAMIC FREQUENCY & DEADLINE LOGIC ---
    frequency = str(task.get("maintenance_frequency", "Ad-hoc")).strip().capitalize()
    frequency_bonus = 0

    if frequency in ["Weekly", "Monthly"]:
        base_freq_bonus = 500 if frequency == "Weekly" else 300
        deadline_str = task.get("deadline_iso")

        if deadline_str:
            try:
                deadline_str = deadline_str.replace("Z", "+00:00")
                deadline_date = datetime.fromisoformat(deadline_str)
                current_time = datetime.now(timezone.utc)

                days_late = (current_time - deadline_date).total_seconds() / 86400.0

                if days_late > 0:
                    # OVERDUE: Polynomial growth (days_late squared)
                    # MULTIPLIER: 200 points per squared day
                    # CEILING: min() caps the bonus at 9,000 so it NEVER beats a Tier 1 emergency
                    penalty = int((days_late ** 2) * 200)
                    frequency_bonus = base_freq_bonus + min(9000, penalty)
                else:
                    # APPROACHING: Slow, linear decay. No exponential panic.
                    days_until = abs(days_late)
                    frequency_bonus = int(base_freq_bonus / (days_until + 1))

            except ValueError:
                frequency_bonus = base_freq_bonus
        else:
            frequency_bonus = base_freq_bonus

    final_score = base_score + aging_bonus + time_impact + frequency_bonus

    if final_score == sys.maxsize:
      return 100000
    else:
      return final_score

# --- EXECUTION ---

if __name__ == "__main__":
    input_file = "mock_simulation_data.json"
    output_file = "delay_counted_prioritized_tasks.json"

    try:
        with open(input_file, "r") as f:
            data = json.load(f)

        queue = data.get("maintenance_queue", data) if isinstance(data, dict) else data

        # Handle list vs dictionary root structure
        if isinstance(queue, dict) and "maintenance_queue" in queue:
            task_list = queue["maintenance_queue"]
        elif isinstance(queue, list):
            task_list = queue
        else:
            task_list = []

        print(f"Loaded {len(task_list)} tasks. Applying priority scoring...\n")

        for task in task_list:
            score = calculate_priority(task)
            task["priority_score"] = score

            display_score = "MAX (Absolute Priority)" if score == sys.maxsize else score
            dept = task.get("department", "Unknown")
            sec = task.get("SectionID", task.get("section_id", "Unknown"))
            tid = task.get("task-id", task.get("tasks_id", "N/A"))
            defect = task.get("defect_type", "Sleeper Replacement")
            tier = task.get("tier", DEFECT_TIER_MAP.get(defect, "Tier 3"))
            freq = str(task.get("maintenance_frequency", "Ad-hoc")).strip().capitalize()

            print(f"Task: {tid:<15} | Tier: {tier:<6} | Freq: {freq:<8} | Section: {sec:<12} | Score: {display_score}")

        with open(output_file, "w") as out_file:
            json.dump(data, out_file, indent=2)

        print(f"\nSuccess! Updated queue saved to '{output_file}'.")

    except FileNotFoundError:
        print(f"Error: '{input_file}' not found. Please generate it first.")

Loaded 30 tasks. Applying priority scoring...

Task: TDMS-2          | Tier: Tier 3 | Freq: Weekly   | Section: HWH_to_BWN   | Score: 1700
Task: TMS-1           | Tier: Tier 1 | Freq: Monthly  | Section: HWH_to_BWN   | Score: 100000
Task: SMMS-1          | Tier: Tier 1 | Freq: Special  | Section: BWN_to_BHP   | Score: 8865
Task: TMS-1           | Tier: Tier 1 | Freq: Special  | Section: BHP_to_MLDT  | Score: 100000
Task: SMMS-3          | Tier: Tier 1 | Freq: Monthly  | Section: KNE_to_NJP   | Score: 9165
Task: TDMS-2          | Tier: Tier 3 | Freq: Monthly  | Section: BWN_to_BHP   | Score: 1500
Task: SMMS-3          | Tier: Tier 1 | Freq: Special  | Section: KNE_to_NJP   | Score: 8865
Task: TDMS-4          | Tier: Tier 1 | Freq: Monthly  | Section: KNE_to_NJP   | Score: 100000
Task: TMS-4           | Tier: Tier 2 | Freq: Monthly  | Section: BWN_to_BHP   | Score: 7207
Task: TMS-3           | Tier: Tier 2 | Freq: Monthly  | Section: HWH_to_BWN   | Score: 7176
Task: TDMS-5          | Tie